In [1]:
import pandas as pd
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"

DISTRICT_OUTPUT = Path("data") / "government_response_district.csv"
BLOCK_OUTPUT = Path("data") / "government_response_block.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# CLEAN KEYS
# =============================================================================

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# PARAMETERS
# =============================================================================

FISCAL_YEAR_START_MONTH = 4

# =============================================================================
# Z-SCORE
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std

# =============================================================================
# CLASSIFICATION (INVERTED)
# =============================================================================

def classify(z):
    if z <= -1.5:
        return 5
    elif z <= -0.5:
        return 4
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 2
    else:
        return 1

# =============================================================================
# FINANCIAL YEAR
# =============================================================================

def get_financial_year(tp):

    year, month = map(int, str(tp).split("_"))

    if month >= FISCAL_YEAR_START_MONTH:
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

# =============================================================================
# DISTRICT-MONTH TENDER TOTAL
# =============================================================================

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(
          district_tender_value=("total_tender_awarded_value", "sum")
      )
)

# =============================================================================
# FINANCIAL YEAR
# =============================================================================

district_df["financial_year"] = district_df["timeperiod"].apply(get_financial_year)

district_df["date"] = pd.to_datetime(
    district_df["timeperiod"],
    format="%Y_%m"
)

district_df = district_df.sort_values(
    ["district", "date"]
)

# =============================================================================
# CUMULATIVE TENDER VALUE
# =============================================================================

district_df["cum_tender_value"] = (
    district_df
    .groupby(["district", "financial_year"])["district_tender_value"]
    .cumsum()
)

# =============================================================================
# MONTHWISE Z-SCORE
# =============================================================================

district_df["govtresponse_z"] = (
    district_df.groupby("timeperiod")["cum_tender_value"]
    .transform(zscore)
)

# =============================================================================
# GOVERNMENT RESPONSE CLASS
# =============================================================================

district_df["government_response"] = (
    district_df["govtresponse_z"]
    .apply(classify)
)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

district_df.to_csv(DISTRICT_OUTPUT, index=False)

# =============================================================================
# APPEND TO BLOCK DATA
# =============================================================================

block_df = df.copy()

if "government_response" in block_df.columns:
    block_df = block_df.drop(columns=["government_response"])

block_df = block_df.merge(
    district_df[
        [
            "district",
            "timeperiod",
            "district_tender_value",
            "cum_tender_value",
            "govtresponse_z",
            "government_response",
        ]
    ],
    on=["district", "timeperiod"],
    how="left",
    validate="many_to_one",
)

# =============================================================================
# SAVE BLOCK OUTPUT
# =============================================================================

block_df.to_csv(BLOCK_OUTPUT, index=False)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\nDistrict output : {DISTRICT_OUTPUT}")
print(f"Block output    : {BLOCK_OUTPUT}")

print("\nDistrict rows:", len(district_df))
print("Block rows:", len(block_df))

print("\nGovernment Response Distribution")
print(district_df["government_response"].value_counts().sort_index())

print("\nMissing values:")
print(block_df["government_response"].isna().sum())

print("\nDistrict preview:")
print(
    district_df[
        [
            "district",
            "timeperiod",
            "district_tender_value",
            "cum_tender_value",
            "govtresponse_z",
            "government_response",
        ]
    ].head()
)

print("\nBlock preview:")
print(
    block_df[
        [
            "object_id",
            "block_name",
            "district",
            "timeperiod",
            "government_response",
        ]
    ].head()
)

Input shape: (92414, 27)

District output : data/government_response_district.csv
Block output    : data/government_response_block.csv

District rows: 690
Block rows: 92414

Government Response Distribution
government_response
1     31
2     18
3    537
4    104
Name: count, dtype: int64

Missing values:
0

District preview:
  district timeperiod  district_tender_value  cum_tender_value  \
0   Anugul    2023_01                    0.0               0.0   
1   Anugul    2023_02                    0.0               0.0   
2   Anugul    2023_03                    0.0               0.0   
3   Anugul    2023_04                    0.0               0.0   
4   Anugul    2023_05                    0.0               0.0   

   govtresponse_z  government_response  
0       -0.208752                    3  
1       -0.215854                    3  
2       -0.240217                    3  
3        0.000000                    3  
4        0.000000                    3  

Block preview:
      object_i